In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.ssl_ import create_urllib3_context
import pandas as pd
import time
import sqlite3
from update_db_func import *
from sqlalchemy import create_engine

In [ ]:
now=pd.Timestamp.now().replace(day=1).strftime('%Y-%m-%d')

In [ ]:
date='2020-01-01'
date_range = pd.date_range(end=date, periods=48, freq='ME')
date_range_str=[int(i.strftime('%Y%m')) for i in date_range]

In [ ]:
class TLSAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        # 手動創建 SSL Context 並指定支援的加密套件
        context = create_urllib3_context()
        # 允許較舊的加密算法 (下放安全性)
        context.set_ciphers('DEFAULT@SECLEVEL=1') 
        kwargs['ssl_context'] = context
        return super(TLSAdapter, self).init_poolmanager(*args, **kwargs)

def get_china_trade_by_country(date):
    s = requests.Session()
    s.mount('https://', TLSAdapter())
    url=r'https://data.mofcom.gov.cn/datamofcom/front/totalbycountry/query'
    headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36 Edg/129.0.0.0',
    }

    playload={'date': date}

    # 之後如常使用 s 進行請求
    response = s.post(url, headers=headers, data=playload)
    time.sleep(5)
    if response.status_code==200:
        data=response.json()['rows']
        df=pd.DataFrame(data)
        return df
    else:
        print('response code: ',response.status_code)
        return pd.DataFrame()

In [ ]:
dfs=[get_china_trade_by_country(i) for i in date_range_str]

In [ ]:
df=pd.concat(dfs)
df['updated_at'] = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')

In [ ]:
s = requests.Session()
s.mount('https://', TLSAdapter())
url=r'https://data.mofcom.gov.cn/datamofcom/front/totalbycountry/query'
headers={'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36 Edg/129.0.0.0',
}

playload={'date': 201901}

# 之後如常使用 s 進行請求
response = s.post(url, headers=headers, data=playload)

In [ ]:
db_file_name=r'china_mocom.db'
db_table_name = r'by_country'
unique_keys = ['trade_date', 'type']
engine = create_engine(f'sqlite:///{db_file_name}')

# create_db(engine, db_table_name, df, unique_keys)
create_update_db(engine, db_table_name, df, unique_keys)

In [ ]:
conn=sqlite3.connect(db_file_name)
df=pd.read_sql_query(f'SELECT * FROM {db_table_name}', conn)
conn.close()
df

In [ ]:
import plotly.express as px

df['date']=pd.to_datetime(df['trade_date'], format='%Y%m')
# df=df.sort_values('date', ascending=False)
df['import_lj_value']=pd.to_numeric(df['import_lj_value'])
df['export_lj_value']=pd.to_numeric(df['export_lj_value'])
df = df.sort_values(['type', 'date'])
df['year'] = df['date'].dt.year
df['import_monthly'] = df.groupby(['year', 'type'])['import_lj_value'].diff()
df['export_monthly'] = df.groupby(['year', 'type'])['export_lj_value'].diff()
df['import_monthly'] = df['import_monthly'].fillna(df['import_lj_value'])
df['export_monthly'] = df['export_monthly'].fillna(df['export_lj_value'])
px.bar(df, x='date', y='import_monthly', color='type', title='中國入口情況')


In [ ]:
px.bar(df, x='date', y='export_monthly', color='type', title='中國出口情況')

In [ ]:
df['import_monthly'] = df['import_monthly'].fillna(df['import_lj_value'])
df['export_monthly'] = df['export_monthly'].fillna(df['export_lj_value'])
df['net_export_import_monthly']=df['export_monthly']-df['import_monthly']
px.bar(df, x='date', y='net_export_import_monthly', color='type', title='中國淨出口情況')